# Auslan → English: extraction on Colab

Runs the pose extraction from `unisign/` on a Colab GPU. Written for the way
Colab actually behaves, not the way a batch job would like it to:

* **Sessions get cut off.** The extraction loop below is built to be killed and
  re-run. Work is checkpointed to Drive in chunks and `extract_pose.py` skips
  clips it has already done, so restarting costs at most one chunk.
* **Drive is slow for many small files.** 76k `.npz` written straight through
  the Drive FUSE mount would dominate the runtime. They are written to local
  Colab disk and shipped to Drive as tar chunks instead.
* **Nothing is ever unpacked.** Both corpora are read directly out of their
  zip archives, so Colab's small local disk is never asked to hold 200 GB.

Measured on a laptop CPU the full job is ~329 hours. The point of this
notebook is to make that a GPU number instead.

> The Uni-Sign weights are **CC BY-NC 4.0**. Anything fine-tuned from them
> inherits the non-commercial restriction.


## 1. Runtime check

If this says CPU, change it: **Runtime → Change runtime type → GPU**. Running
the whole job on a Colab CPU is slower than the laptop it came from.


In [1]:
import subprocess, sys, re

smi = subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,driver_version',
                      '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip() or 'NO GPU')

# The CUDA version nvidia-smi prints is the highest the DRIVER supports, and
# it decides which onnxruntime-gpu build can work here -- see the next cell.
m = re.search(r'CUDA Version:\s*(\d+)\.(\d+)', smi)
CUDA_MAJOR = int(m.group(1)) if m else None
print('driver supports CUDA', f'{m.group(1)}.{m.group(2)}' if m else 'unknown')
print(sys.version)


NVIDIA A100-SXM4-40GB, 40960 MiB, 580.82.07
driver supports CUDA 13.0
3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]


## 2. Dependencies

Three separate traps here. Each one ends with the job running on CPU for days
without ever saying so, which is why the check at the bottom is so insistent.

**`rtmlib` hard-depends on `onnxruntime`** — the CPU-only build. Installed in
the same command as `onnxruntime-gpu` it pulls the CPU package back in, and
since both ship the same `onnxruntime` module directory, whichever lands last
wins. So rtmlib goes in with `--no-deps`.

**The CUDA libraries are `extras`, not dependencies.** Plain
`pip install onnxruntime-gpu` installs *none* of them:

```
nvidia-cuda-runtime~=13.0 ; extra == "cuda"
nvidia-cudnn-cu13~=9.0    ; extra == "cudnn"
```

That is the `libcublasLt.so.13: cannot open shared object file` failure — the
driver supports CUDA 13, but nothing ever installed the CUDA 13 runtime.
`[cuda,cudnn]` fixes it (cuBLAS itself arrives as a dependency of cuDNN).

**onnxruntime-gpu ≥1.27 is built for CUDA 13, ≤1.26 for CUDA 12**, so the
version is chosen from the driver rather than pinned.

Finally, those wheels drop their `.so` files in `site-packages/nvidia/*/lib`,
which is not on the loader path. `preload_dlls()` handles it on recent ORT;
`LD_LIBRARY_PATH` is set as well so the extraction subprocesses inherit it.


In [2]:
import os, glob, subprocess, sys, sysconfig, textwrap

assert CUDA_MAJOR, 'run the runtime-check cell first'
# [cuda,cudnn] are extras and carry the actual CUDA runtime; without them the
# CUDA provider registers but cannot load. <1.27 is the CUDA 12 line.
ORT = ('onnxruntime-gpu[cuda,cudnn]' if CUDA_MAJOR >= 13
       else 'onnxruntime-gpu[cuda,cudnn]<1.27')
print(f'driver CUDA {CUDA_MAJOR}.x -> installing {ORT!r}')

!pip -q uninstall -y onnxruntime onnxruntime-gpu
!pip -q install --no-deps rtmlib
!pip -q install numpy opencv-python-headless tqdm openpyxl
!pip -q install "{ORT}"

# The NVIDIA wheels install into site-packages/nvidia/*/lib, which the dynamic
# loader does not search. Put them on LD_LIBRARY_PATH so every subprocess
# started from this notebook -- including extract_pose.py -- can load them.
libdirs = sorted({d for base in {sysconfig.get_paths()['purelib'],
                                 '/usr/local/lib/python3/dist-packages'}
                    for d in glob.glob(os.path.join(base, 'nvidia', '*', 'lib'))})
if libdirs:
    os.environ['LD_LIBRARY_PATH'] = ':'.join(
        libdirs + [p for p in [os.environ.get('LD_LIBRARY_PATH', '')] if p])
print(f'{len(libdirs)} nvidia lib dirs on LD_LIBRARY_PATH')
print('cublasLt found:', bool(glob.glob(os.path.join(
    sysconfig.get_paths()['purelib'], 'nvidia', '*', 'lib', 'libcublasLt.so*'))))

CHECK = textwrap.dedent('''
    import onnxruntime as ort, numpy as np
    print('onnxruntime', ort.__version__, ort.__file__)
    if hasattr(ort, 'preload_dlls'):
        ort.preload_dlls()          # official way to load the pip CUDA libs
    print('providers  ', ort.get_available_providers())
    if 'CUDAExecutionProvider' not in ort.get_available_providers():
        raise SystemExit('FAIL: CUDA provider not registered')
    from rtmlib import Wholebody
    wb = Wholebody(to_openpose=False, mode='lightweight',
                   backend='onnxruntime', device='cuda')
    for name, m in [('detector', wb.det_model), ('pose', wb.pose_model)]:
        used = m.session.get_providers()
        print(f'{name:9} providers', used)
        if 'CUDAExecutionProvider' not in used:
            raise SystemExit(f'FAIL: {name} session silently fell back to CPU')
    wb(np.zeros((256, 256, 3), dtype=np.uint8))
    print('OK: rtmlib is running on CUDA')
''')

r = subprocess.run([sys.executable, '-c', CHECK], capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    print(r.stderr[-3000:])
    raise RuntimeError('GPU inference is not working -- do not start the '
                       'extraction, it would run on CPU for days. The stderr '
                       'above says why.')


driver CUDA 13.x -> installing 'onnxruntime-gpu[cuda,cudnn]'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.6/69.6 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.3/53.3 MB 45.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 106.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.2/519.2 MB 1.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.8/161.8 MB 14.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.5/61.5 MB 36.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.7/246.7 MB 5.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 440.5/440.5 MB 1.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.5/42.5 MB 51.4 MB/s eta 0:00:00:00:0100:01
18 nvidia lib dirs on LD_LIBRARY_PATH
cublasLt found: True
onnxruntime 1.30.0 /usr/local/lib/python3.13/dist-packages/onnxruntime/__init__.py
provide

## 3. Mount Drive


In [3]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive'


Mounted at /content/drive


## 4. Point at the data

The two datasets live in public Drive folders:

* Auslan-Daily — `17E5wgq1ig7-WynNskG-8tTMoOnGlVh4V`
* MM-WLAuslan — `1EQ1Nh3lidEcu1QLFw0IjRN7YqEq1N48q`

**Add them to your own Drive before running this** (open the folder in a
browser → *Organise* → *Add shortcut to Drive*). A shortcut is enough and
costs no storage. If a heavily-shared file starts returning quota errors on
read, make a real copy instead — a file you own has no public download cap.
That cap is what stopped the News archive downloading earlier, so it is not a
hypothetical.

The cell below finds the datasets by their contents rather than by a path you
have to type, and says what it found.


In [4]:
import os, glob

# Bounded search, not glob('**'): a recursive glob over a mounted Drive walks
# every folder you own through a FUSE layer and can take minutes.
SEARCH_ROOTS = [DRIVE, '/content/drive/Shareddrives', '/content']

def _find_file(names, max_depth=6):
    for root in SEARCH_ROOTS:
        if not os.path.isdir(root):
            continue
        for depth in range(max_depth):
            for name in names:
                for hit in glob.glob(os.path.join(root, *(['*'] * depth), name)):
                    return hit
    return None

def _walk_up_to(hit, marker):
    """Climb from a file to the ancestor that CONTAINS `marker`.

    Counting directory levels down from a fixed pattern is brittle -- the
    release nests the translation tables one level deeper than its own docs
    suggest, in folders spelled 'Ausan-Daily ...'. Walking up to a landmark
    does not care how deep it is or how it is spelled.
    """
    d = os.path.dirname(hit)
    for _ in range(8):
        if os.path.isdir(os.path.join(d, marker)):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            break
        d = parent
    return None

AD_ROOT = MMWL_ROOT = None

hit = _find_file(['Auslan-Daily_News.xlsx', 'Auslan-Daily_Communication.xlsx'])
if hit:
    AD_ROOT = _walk_up_to(hit, 'Dataset Split Table')
print('Auslan-Daily:', AD_ROOT or 'NOT FOUND')

hit = _find_file(['Train.json'])
if hit:
    MMWL_ROOT = (_walk_up_to(hit, 'Annotation')
                 or _walk_up_to(hit, 'labels'))
print('MM-WLAuslan :', MMWL_ROOT or 'NOT FOUND')

if not (AD_ROOT and MMWL_ROOT):
    print(f'\ntop level of {DRIVE}:')
    for name in sorted(os.listdir(DRIVE))[:40]:
        p = os.path.join(DRIVE, name)
        try:
            n = f'{len(os.listdir(p))} items' if os.path.isdir(p) else 'file'
        except OSError as e:
            n = f'UNREADABLE ({e.strerror})'
        print(f'  {name}   {n}')


Auslan-Daily: /content/drive/MyDrive/Auslan-Daily
MM-WLAuslan : /content/drive/MyDrive/MM-WLAuslan


## 5. The code

Three ways to get `unisign/` here, in order of convenience:

1. copy the folder into your Drive (survives every session)
2. VS Code Explorer → right-click `unisign/` → **Upload to Colab**
3. run the cell with nothing in place and it will prompt you to upload
   `unisign_code.tar.gz` — 49 KB, in the project root

The fingerprint printed below **must** match what your laptop prints
(`python unisign/spec.py`). It hashes the keypoint indices, roots and
confidence threshold — the things that must be identical to Uni-Sign's
pre-training and identical between the two corpora. If it differs, `.npz`
produced here cannot be mixed with `.npz` produced locally, and nothing
downstream would tell you.


In [5]:
import glob, os, shutil, sys, tarfile

CODE = '/content/unisign'

def locate_code():
    for root in [DRIVE, '/content']:
        for prefix in ['', '*/', '*/*/']:
            for hit in glob.glob(os.path.join(root, prefix, 'unisign', 'spec.py')):
                return os.path.dirname(hit)
    return None

def locate_tarball():
    for root in [DRIVE, '/content']:
        for prefix in ['', '*/']:
            hits = glob.glob(os.path.join(root, prefix, 'unisign_code.tar.gz'))
            if hits:
                return hits[0]
    return None

SRC = locate_code()
if SRC is None:
    tarball = locate_tarball()
    if tarball is None:
        print('unisign/ not found. Upload unisign_code.tar.gz (49 KB, in the '
              'project root) when the picker appears.')
        from google.colab import files
        up = files.upload()
        tarball = next(iter(up))
    print('extracting', tarball)
    with tarfile.open(tarball) as tf:
        tf.extractall('/content')
    SRC = locate_code()

assert SRC, 'still no unisign/spec.py -- check what the upload produced'
if os.path.abspath(SRC) != CODE:
    shutil.rmtree(CODE, ignore_errors=True)
    shutil.copytree(SRC, CODE)
print('code at', CODE, '(from', SRC + ')')

sys.path.insert(0, CODE)
import spec
print()
print(spec.describe())
print()
print('fingerprint MUST equal the local one:', spec.SCHEMA_FINGERPRINT)


code at /content/unisign (from /content/drive/MyDrive/unisign)

spec=unisign-pose-v2-verified fingerprint=bc3bb2df0f22948d
channels per joint: 3 (x, y, confidence)   conf threshold: 0.3
shared encoders: (('left', 'right'),)
  body      n=  9  root=crop_scale, no root
    0-based wholebody indices: [0, 3, 4, 5, 6, 7, 8, 9, 10]
  left      n= 21  root=local 0 = wholebody 91 (0-based)
    0-based wholebody indices: [91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111]
  right     n= 21  root=local 0 = wholebody 112 (0-based)
    0-based wholebody indices: [112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132]
  face_all  n= 18  root=local -1 = wholebody 53 (0-based)
    0-based wholebody indices: [23, 25, 27, 29, 31, 33, 35, 37, 39, 83, 84, 85, 86, 87, 88, 89, 90, 53]

fingerprint MUST equal the local one: bc3bb2df0f22948d


## 6. Build the manifest

One MM-WLAuslan manifest for this extraction run, every clip referenced as
`<archive>::<member>`. Nothing is unpacked.

This run is for MM-WLAuslan pose extraction only. The three frontal studio
splits are enabled: `Train`, `Valid`, and `Test_STU`. Auslan-Daily is not
added to this manifest or copied to Colab. `Test_MTV` remains excluded
because it is a multi-device set rather than the frontal `kf` camera.


In [6]:
import glob, json, os, subprocess, sys

if not (AD_ROOT or MMWL_ROOT):
    raise SystemExit('Neither dataset was found -- fix the previous cell first.')

WORK = f'{DRIVE}/auslan_work'
os.makedirs(WORK, exist_ok=True)

# This extraction run is deliberately MM-WLAuslan-only. Auslan-Daily
# poses are already in Drive and must not cause its large video archives
# to be copied to Colab again.
USE_AUSLAN_DAILY = False
MMWL_SPLITS = ['Train', 'Valid', 'Test_STU']

def run(script, args):
    """Run a pipeline script as a subprocess so its message survives."""
    r = subprocess.run([sys.executable, script] + args, cwd=CODE,
                       capture_output=True, text=True)
    print(r.stdout)
    if r.returncode != 0:
        print(r.stderr[-3000:])
        raise RuntimeError(f'{script} failed')

def find_one(root, pattern, what):
    """Locate a file under `root` without assuming how deep it sits.

    The release nests the translation tables inside per-subset folders spelled
    'Ausan-Daily Communication' / 'Ausan-Daily News' -- a level deeper than its
    own documentation shows, and misspelled. Hard-coding the path is how that
    turns into an empty manifest.
    """
    for depth in range(6):
        hits = glob.glob(os.path.join(root, *(['*'] * depth), pattern))
        if hits:
            return sorted(hits)[0]
    raise SystemExit(f'could not find {what} ({pattern}) under {root}')

records = f'{WORK}/auslan_daily_records.json'
if USE_AUSLAN_DAILY and AD_ROOT:
    ad_args = []
    for subset, table, folder in [
            ('communication', 'Auslan-Daily_Communication.xlsx', 'Communication'),
            ('news', 'Auslan-Daily_News.xlsx', 'News')]:
        tbl = find_one(AD_ROOT, table, f'{subset} split table')
        zipf = find_one(os.path.join(AD_ROOT, f'*{folder}'),
                        'Signer.zip', f'{subset} Signer-Only clips')
        print(f'{subset}:\n  table {tbl}\n  clips {zipf}')
        ad_args += ['--split-table', tbl, '--video-root', zipf,
                    '--subset', subset]
    run('auslan_daily.py', ad_args + ['--out', records])

# Keep this separate from the combined manifest used later by B/C training.
MANIFEST = f'{WORK}/manifest_mmwlauslan.jsonl'
args = ['--out', MANIFEST]
if MMWL_ROOT and MMWL_SPLITS:
    args += ['--mmwl-root', MMWL_ROOT, '--mmwl-camera', 'kf',
             '--mmwl-splits'] + MMWL_SPLITS
if USE_AUSLAN_DAILY and AD_ROOT:
    args += ['--ad-annotations', records]
if len(args) == 2:
    raise SystemExit('Nothing selected: MM-WLAuslan was not found or its '
                     'splits are empty. Fix the dataset cell above.')
run('manifest.py', args)


51440 clips
  mmwlauslan   studio         test      6430
  mmwlauslan   studio         train    38580
  mmwlauslan   studio         val       6430
  distinct glosses: 3215
-> /content/drive/MyDrive/auslan_work/manifest_mmwlauslan.jsonl



## 7. Stage the archives on local disk

**This is the difference between 32 clips/min and a working GPU.**

Reading clips straight out of the archives on Drive means every clip is a seek
and a read into a multi-gigabyte zip across a FUSE network mount. With eight
workers doing that at once the GPU spends its time waiting: measured at 5.3
frames/s per worker, which is *slower than one process on a laptop CPU*.

Copying each archive to Colab's local disk once, up front, moves every
subsequent read to local SSD. The copy is a few minutes; the saving is hours.

Nothing is unpacked — the archives stay zipped and are still read in place,
just from a disk instead of from the network. The manifest is rewritten to
point at the local copies.


In [7]:
import json, os, shutil, time

LOCAL_ARCHIVES = '/content/archives'
os.makedirs(LOCAL_ARCHIVES, exist_ok=True)

rows = [json.loads(l) for l in open(MANIFEST)]
archives = sorted({r['video'].split('::')[0] for r in rows if '::' in r['video']})

if archives and all(a.startswith(LOCAL_ARCHIVES + '/') for a in archives):
    # Re-running this cell after it succeeded: nothing to copy, and renaming
    # the local paths again would stage a second copy under a mangled name.
    print(f'MANIFEST already points at local archives -> {MANIFEST}')
else:
    def local_name(a):
        # Several archives share a basename (both Auslan-Daily subsets ship
        # Signer.zip; MM-WLAuslan ships rgb.zip per split), so name each copy
        # after its last three path components.
        parts = os.path.normpath(a).split(os.sep)[-3:]
        return os.path.join(LOCAL_ARCHIVES, '__'.join(parts).replace(' ', '_'))

    remap = {a: local_name(a) for a in archives}
    if len(set(remap.values())) != len(remap):
        raise SystemExit(f'local names collide: {remap}')

    # Decide what is missing by the SAME names used for copying -- checking a
    # different name than the one written is what made a re-run think all of
    # it still needed copying, and then refuse for lack of disk.
    todo = [a for a in archives
            if not (os.path.exists(remap[a]) and os.path.getsize(remap[a]) == os.path.getsize(a))]
    need = sum(os.path.getsize(a) for a in todo)
    free = shutil.disk_usage(LOCAL_ARCHIVES).free
    print(f'{len(archives)} archive(s): {len(archives) - len(todo)} already local, '
          f'{len(todo)} to copy ({need/1e9:.1f} GB), {free/1e9:.1f} GB free')
    if need > free * 0.9:
        raise SystemExit('Not enough local disk for the MM-WLAuslan archives '
                         'and pose files -- free space under /content.')

    for a in archives:
        if a not in todo:
            print(f'  have   {os.path.basename(remap[a])}')
            continue
        t = time.time()
        shutil.copy(a, remap[a])
        dt = time.time() - t
        gb = os.path.getsize(remap[a]) / 1e9
        print(f'  copied {os.path.basename(remap[a])}  {gb:.1f} GB in {dt/60:.1f} min '
              f'({gb * 1e3 / max(dt, 1):.0f} MB/s)')

    MANIFEST_LOCAL = MANIFEST.replace('.jsonl', '_local.jsonl')
    with open(MANIFEST_LOCAL, 'w') as fh:
        for r in rows:
            if '::' in r['video']:
                a, m = r['video'].split('::', 1)
                r['video'] = f'{remap[a]}::{m}'
            fh.write(json.dumps(r, ensure_ascii=False) + '\n')
    MANIFEST = MANIFEST_LOCAL          # everything downstream uses the local copies

first = json.loads(open(MANIFEST).readline())['video'].split('::')[0]
print(f'\nMANIFEST = {MANIFEST}\nfirst clip reads from: {first}')
if not first.startswith(LOCAL_ARCHIVES + '/'):
    raise SystemExit('Still pointing at Drive -- do not start the extraction.')


3 archive(s): 0 already local, 3 to copy (19.6 GB), 65.9 GB free
  copied MM-WLAuslan__Test_STU__rgb.zip  2.6 GB in 0.9 min (50 MB/s)
  copied MM-WLAuslan__Train__rgb.zip  14.5 GB in 4.6 min (53 MB/s)
  copied MM-WLAuslan__Valid__rgb.zip  2.4 GB in 0.5 min (74 MB/s)

MANIFEST = /content/drive/MyDrive/auslan_work/manifest_mmwlauslan_local.jsonl
first clip reads from: /content/archives/MM-WLAuslan__Train__rgb.zip


## 8. Workspace and previous progress

`.npz` are written to local Colab disk and shipped to Drive as tar chunks.
This run has 51,440 MM-WLAuslan clips. Any older Auslan-Daily `.npz` files
restored into the same local directory are not part of this manifest.
Writing tens of thousands of small files straight through the Drive FUSE
mount would dominate the runtime; one tar every few minutes does not.

This cell pulls back only MM-WLAuslan files from whatever earlier sessions
finished, so this run resumes without restoring old Auslan-Daily `.npz`
files into the local disk.


In [8]:
import glob, json, os, tarfile, shutil, time, subprocess, sys

POSE_LOCAL = '/content/pose'
POSE_DRIVE = f'{WORK}/pose'
LOGS = '/content/worker_logs'
for d in (POSE_LOCAL, POSE_DRIVE, LOGS):
    os.makedirs(d, exist_ok=True)

manifest_uids = [json.loads(line)['uid'] for line in open(MANIFEST)
                 if line.strip()]
wanted_names = {f'{uid}.npz' for uid in manifest_uids}
chunks = sorted(glob.glob(f'{POSE_DRIVE}/chunk_*.tar'))
archived = set()
for c in chunks:
    with tarfile.open(c) as tf:
        for member in tf.getmembers():
            if member.name in wanted_names:
                tf.extract(member, POSE_LOCAL)
                archived.add(member.name)

total = len(manifest_uids)
def count_present():
    return sum(os.path.exists(os.path.join(POSE_LOCAL, f'{uid}.npz'))
               for uid in manifest_uids)
have = count_present()
print(f'manifest {total} clips | already extracted {have} '
      f'(restored from {len(chunks)} chunk(s))')


/tmp/ipykernel_496/1504162318.py:18: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extract(member, POSE_LOCAL)


manifest 51440 clips | already extracted 45443 (restored from 445 chunk(s))


## 9. Extract — several workers on one GPU

The models are small (19 MB detector, 123 MB pose) and rtmlib runs **one frame
at a time**, so a single process leaves an A100 almost idle: about 1 GB of
40 GB, with the real bottleneck being per-frame Python and video decoding
rather than GPU maths. Low memory use is not the same as low utilisation, but
here both are low, and the fix is concurrency rather than a bigger card.

So this runs `NUM_WORKERS` processes over disjoint shards of the manifest.
They cannot collide: each clip's `.npz` is named by its `uid` and every uid
belongs to exactly one shard.

Tune `NUM_WORKERS` by watching `nvidia-smi` while it runs. Going up stops
helping once video decoding saturates the vCPUs, which usually happens before
GPU memory does.

**This cell is meant to be interrupted.** Progress is checkpointed to Drive, so
a killed session — or Stop — costs at most one checkpoint interval. Re-running
sections 1–8 in a fresh session and then this one continues where it left off.

The cell kills leftover workers before starting, so re-running it after a
Stop cannot end up with two sets of workers fighting over the same vCPUs.


In [9]:
# Kill anything left over from an earlier run FIRST.
#
# Pressing Stop on this cell raises KeyboardInterrupt in the notebook, but a
# worker that ignores SIGTERM, or a cell killed outright, can leave processes
# alive and holding GPU memory. Re-running would then start a second full set
# of workers competing with the first -- twice the video decoding on the same
# vCPUs, for no extra throughput. They also all write to the same directory,
# so it is not corrupting, just slower and confusing.
import subprocess
stray = subprocess.run(['pgrep', '-f', 'extract_pose.py'],
                       capture_output=True, text=True).stdout.split()
if stray:
    print(f'killing {len(stray)} leftover worker(s): {stray}')
    subprocess.run(['pkill', '-9', '-f', 'extract_pose.py'])
    time.sleep(3)
else:
    print('no leftover workers')
print(subprocess.run(['nvidia-smi','--query-gpu=memory.used,utilization.gpu',
                      '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip())

NUM_WORKERS = 14            # ~1 GB GPU each; raise until nvidia-smi stops improving
REPORT_SECONDS = 60        # how often progress is printed
CHECKPOINT_SECONDS = 300   # how often new .npz are tarred to Drive

# Reporting and checkpointing are separate on purpose. Tarring to Drive every
# minute would waste time; but waiting five minutes for the first sign of life
# leaves you unable to tell a working run from a stuck one.

def checkpoint():
    """Tar the .npz Drive does not have yet. One file, not thousands."""
    new = [f'{uid}.npz' for uid in manifest_uids
           if f'{uid}.npz' not in archived
           and os.path.exists(os.path.join(POSE_LOCAL, f'{uid}.npz'))]
    if not new:
        return 0
    n = len(glob.glob(f'{POSE_DRIVE}/chunk_*.tar'))
    tmp = '/content/_chunk.tar'
    with tarfile.open(tmp, 'w') as tf:
        for name in new:
            tf.add(f'{POSE_LOCAL}/{name}', arcname=name)
    shutil.copy(tmp, f'{POSE_DRIVE}/chunk_{n:04d}.tar')
    os.remove(tmp)
    archived.update(new)
    return len(new)

procs, logs = [], []
for i in range(NUM_WORKERS):
    log = open(f'{LOGS}/worker{i}.log', 'w')
    logs.append(log)
    procs.append(subprocess.Popen(
        [sys.executable, 'extract_pose.py',
         '--manifest', MANIFEST, '--out', POSE_LOCAL, '--device', 'cuda',
         '--shard', str(i), '--num-shards', str(NUM_WORKERS)],
        cwd=CODE, stdout=log, stderr=subprocess.STDOUT))
print(f'{NUM_WORKERS} workers started; logs in {LOGS}')

start_n = count_present()
t0 = time.time()
try:
    last_ckpt = time.time()
    while any(p.poll() is None for p in procs):
        time.sleep(REPORT_SECONDS)
        saved = 0
        if time.time() - last_ckpt >= CHECKPOINT_SECONDS:
            saved = checkpoint()
            last_ckpt = time.time()
        n = count_present()
        el = time.time() - t0
        rate = (n - start_n) / max(el, 1e-9)
        eta = (total - n) / rate / 3600 if rate > 0 else float('inf')
        alive = sum(p.poll() is None for p in procs)
        gpu = subprocess.run(['nvidia-smi', '--query-gpu=utilization.gpu',
                              '--format=csv,noheader'],
                             capture_output=True, text=True).stdout.strip()
        print(f'{n}/{total} | {rate*60:.1f} clip/min | ~{eta:.1f} h left '
              f'| gpu {gpu} | {alive}/{NUM_WORKERS} workers'
              + (f' | +{saved} to Drive' if saved else ''), flush=True)
finally:
    # Runs on interrupt too, so stopping the cell never loses the last window.
    for p in procs:
        if p.poll() is None:
            p.terminate()
    for f in logs:
        f.close()
    print('checkpointing before exit ...')
    print(f'+{checkpoint()} clips saved to Drive')

bad = [i for i, p in enumerate(procs) if p.returncode not in (0, None, -15)]
if bad:
    print(f'\nworkers {bad} exited with an error; last lines:')
    for i in bad:
        print(f'--- worker{i} ---')
        print(open(f'{LOGS}/worker{i}.log').read()[-1500:])
else:
    print(f'\n{count_present()}/{total} MM-WLAuslan clips extracted')


no leftover workers
0 MiB, 0 %
14 workers started; logs in /content/worker_logs
45457/51440 | 13.9 clip/min | ~7.2 h left | gpu 100 % | 14/14 workers
45493/51440 | 24.7 clip/min | ~4.0 h left | gpu 100 % | 14/14 workers
45526/51440 | 27.4 clip/min | ~3.6 h left | gpu 100 % | 14/14 workers
45559/51440 | 28.7 clip/min | ~3.4 h left | gpu 100 % | 14/14 workers
45591/51440 | 29.3 clip/min | ~3.3 h left | gpu 100 % | 14/14 workers | +148 to Drive
45627/51440 | 30.4 clip/min | ~3.2 h left | gpu 100 % | 14/14 workers
45660/51440 | 30.7 clip/min | ~3.1 h left | gpu 100 % | 14/14 workers
45697/51440 | 31.5 clip/min | ~3.0 h left | gpu 100 % | 14/14 workers
45733/51440 | 31.9 clip/min | ~3.0 h left | gpu 100 % | 14/14 workers
45768/51440 | 32.2 clip/min | ~2.9 h left | gpu 100 % | 14/14 workers | +177 to Drive
45799/51440 | 32.0 clip/min | ~2.9 h left | gpu 100 % | 14/14 workers
45833/51440 | 32.2 clip/min | ~2.9 h left | gpu 100 % | 14/14 workers
45870/51440 | 32.5 clip/min | ~2.9 h left | gpu 

## 10. The gate

Same three-layer check as locally: that every clip came from one spec and one
model, the per-clip quality flags, and the mirror test. **Do not start
training on a set this reports problems for** — a keypoint set that is
internally inconsistent does not raise anywhere downstream, it just produces a
disappointing BLEU weeks later.


In [10]:
import subprocess, sys
r = subprocess.run([sys.executable, 'verify_pose.py', '--npz-dir', POSE_LOCAL,
                    '--csv', f'{WORK}/quality.csv',
                    '--exclude-out', f'{WORK}/excluded.txt',
                    '--overlay-dir', f'{WORK}/overlays_mirrored',
                    '--overlay-flag', 'MIRRORED_OR_BACK_VIEW', '--overlay-n', '131'],
                   cwd=CODE, capture_output=True, text=True)
print(r.stdout[-8000:])
if r.returncode != 0:
    print(r.stderr[-2000:])
    raise RuntimeError('verify_pose reported problems -- do not train on this set')



=== layer 1: consistency ===
  OK  spec=unisign-pose-v2-verified fp=bc3bb2df0f22948d
      model=rtmlib.Wholebody/lightweight backend=onnxruntime rtmlib=0.0.16 person=largest

=== layer 2: numbers ===
  clips=51440
  mmwlauslan/studio          n= 51440  dominant-hand present median=1.000
  out_of_bounds                  17  (0.0%)

  0 clips excluded -> /content/drive/MyDrive/auslan_work/excluded.txt

  per-clip stats -> /content/drive/MyDrive/auslan_work/quality.csv

=== layer 3: eyes ===

  Watch these. Orange must be on the signer's left hand.



## When the session dies

Re-run sections 1 → 9 in order in the new session.

- **Section 7** copies the archives back to local disk. A fresh runtime's
  `/content` is empty, so this takes about 9 minutes again (measured at
  54–58 MB/s from Drive).
- **Section 8** pulls every checkpointed chunk back from Drive.
- **Section 9** skips clips that are already there and carries on. At most one
  checkpoint interval (`CHECKPOINT_SECONDS`, 5 minutes) of work is lost.

Parallelism lives inside one session: raise `NUM_WORKERS` in section 9 rather
than opening more sessions. Keep it at or below the vCPU count
(`os.cpu_count()`); beyond that, throughput falls.

**Re-extracting under a new policy or spec** — for example after the default
`person_select` changed from `track` to `largest` — rename the Drive checkpoint
folder (`auslan_work/pose` → e.g. `pose_track_old`) before running section 8.
Otherwise section 8 restores the old clips. Renaming keeps the old data rather
than deleting it.

When extraction finishes, section 10 checks the whole set. Only once it passes,
pull the chunks back to the laptop and untar them into `data/pose/`.
